In [4]:
import sys, os, re, json, shutil, subprocess, warnings
sys.path.append("../pipelines")
from aq import *
import yaml
warnings.filterwarnings("ignore")

RACINE = "/home/jovyan/work" if os.path.isdir("/home/jovyan/work/soda") else "."
SODA = f"{RACINE}/soda"
os.makedirs(f"{SODA}/checks/bronze", exist_ok=True)
os.makedirs(f"{SODA}/reports", exist_ok=True)
SODA_BIN = shutil.which("soda") or os.path.join(os.path.dirname(sys.executable), "soda")
REGLES = lire("SELECT rule_id, severite, seuil_min FROM dq.regle").set_index("rule_id")
CODES_RETOUR = {0: "conforme", 1: "avertissement", 2: "échec", 3: "erreur d'exécution"}

def nombre(v):
    return int(v) if isinstance(v, float) and v.is_integer() else v

def scan(source, fichier):
    sortie = f"{SODA}/reports/{source}_{os.path.basename(fichier).replace('.yml', '')}.json"
    if os.path.exists(sortie):
        os.remove(sortie)
    p = subprocess.run([SODA_BIN, "scan", "-d", source, "-c", f"{SODA}/configuration.yml", "-srf", sortie, fichier], capture_output=True, text=True)
    if not os.path.exists(sortie):
        raise RuntimeError(f"Scan en erreur (code {p.returncode}) :\n{p.stdout[-1500:]}\n{p.stderr[-800:]}")
    controles = []
    for c in json.load(open(sortie, encoding="utf-8"))["checks"]:
        regle = re.search(r"RQ-\d{3}", c["name"])
        controles.append({"fichier": os.path.basename(fichier), "couche": source, "rule_id": regle.group(0) if regle else None,
                          "controle": c["name"], "resultat": c["outcome"], "valeur": nombre((c.get("diagnostics") or {}).get("value"))})
    return p.returncode, controles

fmt = lambda n: f"{int(n):,}".replace(",", " ")

In [5]:
print("Sources de données déclarées :", [k.replace("data_source ", "") for k in yaml.safe_load(open(f"{SODA}/configuration.yml", encoding="utf-8"))])

TABLES_BRONZE = {"client": "crm_client", "contrat": "crm_contrat", "sinistre": "crm_sinistre", "type_contrat": "crm_type_contrat"}
def vers_bronze(texte):
    for source, cible in TABLES_BRONZE.items():
        texte = re.sub(rf"(checks for |filter |FROM |JOIN |exist in )({source})\b", rf"\1{cible}", texte)
    return texte.replace("[12, 24, 36, 60]", "['12', '24', '36', '60']")

for nom in ["client", "contrat", "sinistre"]:
    texte = open(f"{SODA}/checks/{nom}_checks.yml", encoding="utf-8").read()
    open(f"{SODA}/checks/bronze/{nom}_checks.yml", "w", encoding="utf-8").write(vers_bronze(texte))

open(f"{SODA}/checks/bronze/integrite_checks.yml", "w", encoding="utf-8").write("""# RQ-016 : un souscripteur a au moins un contrat (règle d'avertissement dans dq.regle, seuil 98 %)
checks for crm_client:
  - failed rows:
      name: RQ-016 souscripteur sans contrat
      fail query: |
        SELECT k.client_id FROM crm_client k
        WHERE k.type_personne = 'souscripteur'
          AND NOT EXISTS (SELECT 1 FROM crm_contrat c WHERE c.souscripteur_id = k.client_id)
      warn: when > 0

# Relation sinistre vers contrat : aucune clé étrangère n'est déclarée dans la base
checks for crm_sinistre:
  - values in (contrat_id) must exist in crm_contrat (contrat_id):
      name: Relation sinistre vers contrat
""")

CONTROLES, CODES = [], {}
for nom in ["client_checks", "contrat_checks", "sinistre_checks", "integrite_checks"]:
    CODES[nom], c = scan("bronze", f"{SODA}/checks/bronze/{nom}.yml")
    CONTROLES += c
CTRL = pd.DataFrame(CONTROLES, dtype=object)
CTRL["severite"] = CTRL.rule_id.map(REGLES.severite)
print(CTRL[["fichier", "rule_id", "controle", "resultat", "valeur", "severite"]].sort_values(["fichier", "rule_id"]).to_string(index=False))

print("\nCodes retour par fichier :", {k: f"{v} ({CODES_RETOUR[v]})" for k, v in CODES.items()})
code_porte = max(CODES.values())
bloquants = CTRL[(CTRL.resultat == "fail") & (CTRL.severite == "BLOQUANTE")].rule_id.dropna().unique().tolist()
print(f"Code retour de la porte : {code_porte} ({CODES_RETOUR[code_porte]}) ; règles bloquantes en échec : {bloquants}")
print("Avertissements :", CTRL[CTRL.resultat == "warn"].controle.tolist())

Sources de données déclarées : ['crm', 'bronze', 'silver', 'gold']
             fichier rule_id                                             controle resultat valeur      severite
   client_checks.yml  RQ-003           RQ-003 telephone du souscripteur renseigne     fail   5989 AVERTISSEMENT
   client_checks.yml  RQ-005           RQ-005 code postal sur 5 chiffres (France)     fail   1456 AVERTISSEMENT
   client_checks.yml  RQ-007               RQ-007 pays dans la liste de reference     fail   3050     BLOQUANTE
   client_checks.yml  RQ-009                     RQ-009 type de personne autorise     fail    200     BLOQUANTE
   client_checks.yml  RQ-015                         RQ-015 prospect sans contrat     fail    479     BLOQUANTE
   client_checks.yml    None                       duplicate_count(client_id) = 0     pass      0           NaN
   client_checks.yml    None                         missing_count(client_id) = 0     pass      0           NaN
  contrat_checks.yml  RQ-001    RQ-00

In [6]:
RESULTAT = {"pass": "PASSED", "warn": "WARN", "fail": "FAILED"}

def regle_affichee(r):
    if r.rule_id:
        return r.rule_id
    if r.controle.startswith(("duplicate_count", "missing_count(client_id", "missing_count(contrat_id", "missing_count(sinistre_id")):
        return "RQ-020"
    if "adresse_sinistre" in r.controle or r.controle.startswith("taux_erreur_adresse"):
        return "RQ-014"
    if r.controle.startswith("ecart_client_actif"):
        return "RQ-018"
    return "sans RQ"

tab = CTRL.copy()
tab["regle"] = [regle_affichee(r) for r in tab.itertuples(index=False)]
tab["code"] = [CODES[f.replace(".yml", "")] for f in tab.fichier]
tab = tab.sort_values(["regle", "fichier"])
print("| Règle | Contrôle | Fichier de checks | Couche mesurée | Résultat Soda (PASSED, WARN, FAILED) | Valeur mesurée | Code retour du scan |")
print("|---|---|---|---|---|---|---|")
for r in tab.itertuples(index=False):
    print(f"| {r.regle} | {r.controle} | `soda/checks/bronze/{r.fichier}` | BRONZE | {RESULTAT.get(r.resultat, r.resultat)} | {r.valeur} | {r.code} ({CODES_RETOUR[r.code]}) |")

s2 = mesures("BRONZE")
s2 = s2[s2.rule_id.isin(["RQ-001", "RQ-003", "RQ-005", "RQ-007", "RQ-009", "RQ-010"])][["rule_id", "table_cible", "nb_echecs"]]
soda = tab[tab.regle.isin(s2.rule_id)].groupby("regle").valeur.apply(lambda v: int(pd.to_numeric(v, errors="coerce").sum())).rename("echecs_soda")
comparaison = s2.rename(columns={"nb_echecs": "echecs_great_expectations"}).merge(soda, left_on="rule_id", right_index=True, how="left")
comparaison["ecart"] = comparaison.echecs_soda - comparaison.echecs_great_expectations.astype(int)
print("\nComparaison avec Great Expectations (S2) :")
print(comparaison.to_string(index=False))

| Règle | Contrôle | Fichier de checks | Couche mesurée | Résultat Soda (PASSED, WARN, FAILED) | Valeur mesurée | Code retour du scan |
|---|---|---|---|---|---|---|
| RQ-001 | RQ-001 date de fin posterieure a la date de debut | `soda/checks/bronze/contrat_checks.yml` | BRONZE | FAILED | 800 | 2 (échec) |
| RQ-003 | RQ-003 telephone du souscripteur renseigne | `soda/checks/bronze/client_checks.yml` | BRONZE | FAILED | 5989 | 2 (échec) |
| RQ-005 | RQ-005 code postal sur 5 chiffres (France) | `soda/checks/bronze/client_checks.yml` | BRONZE | FAILED | 1456 | 2 (échec) |
| RQ-007 | RQ-007 pays dans la liste de reference | `soda/checks/bronze/client_checks.yml` | BRONZE | FAILED | 3050 | 2 (échec) |
| RQ-009 | RQ-009 type de personne autorise | `soda/checks/bronze/client_checks.yml` | BRONZE | FAILED | 200 | 2 (échec) |
| RQ-010 | RQ-010 duree de garantie autorisee | `soda/checks/bronze/contrat_checks.yml` | BRONZE | FAILED | 600 | 2 (échec) |
| RQ-011 | RQ-011 type de contrat connu | `sod

In [8]:
def valeur(rule_id=None, controle=None):
    ligne = CTRL[CTRL.rule_id == rule_id] if rule_id else CTRL[CTRL.controle == controle]
    return int(ligne.valeur.iloc[0])
def total(requete):
    return int(lire(requete, b=BATCH).iloc[0, 0])
RELATIONS = pd.DataFrame([
    ("RQ-007 · client.pays -> ref_pays",                    total("SELECT COUNT(*) FROM bronze.crm_client WHERE _batch_id = :b"),  valeur("RQ-007")),
    ("RQ-011 · contrat.code_type -> type_contrat",          total("SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b"), valeur("RQ-011")),
    ("RQ-012 · contrat.souscripteur_id -> client",          total("SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b"), valeur("RQ-012")),
    ("RQ-013 · contrat.assure_id -> client (sentinelle 0 exclue)", total("SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b AND assure_id <> '0'"), valeur("RQ-013")),
    ("sinistre.contrat_id -> contrat",                      total("SELECT COUNT(*) FROM bronze.crm_sinistre WHERE _batch_id = :b"), valeur(controle="Relation sinistre vers contrat"))],
    columns=["relation", "lignes_testees", "orphelins"])
RELATIONS["pct"] = (100 * RELATIONS.orphelins / RELATIONS.lignes_testees).round(3)
print(RELATIONS.to_string(index=False))

orphelins_souscripteur = lire("""SELECT c.contrat_id, c.souscripteur_id FROM bronze.crm_contrat c
    LEFT JOIN bronze.crm_client k ON k.client_id = c.souscripteur_id AND k._batch_id = :b
    WHERE c._batch_id = :b AND k.client_id IS NULL LIMIT 20""", b=BATCH)
print(orphelins_souscripteur.head(5).to_string(index=False))

                                                  relation  lignes_testees  orphelins  pct
                          RQ-007 · client.pays -> ref_pays          100000       3050 3.05
                RQ-011 · contrat.code_type -> type_contrat          100000        200 0.20
                RQ-012 · contrat.souscripteur_id -> client          100000       1000 1.00
RQ-013 · contrat.assure_id -> client (sentinelle 0 exclue)           60000        300 0.50
                            sinistre.contrat_id -> contrat          100000          0 0.00
contrat_id souscripteur_id
 CT0000031         H-97935
 CT0000046         S-97321
 CT0000071         S-95441
 CT0000072         S-97658
 CT0000188         A-95721


In [9]:
print("| Relation | Lignes testées | Orphelins | Taux | Conclusion (anomalie, sentinelle, normal) |")
print("|---|---|---|---|---|")
for r in RELATIONS.itertuples(index=False):
    print(f"| {r.relation} | {fmt(r.lignes_testees)} | {fmt(r.orphelins)} | {r.pct:.3f} % | … |")

detail = {}
detail["souscripteurs orphelins distincts"] = total("""SELECT COUNT(DISTINCT c.souscripteur_id) FROM bronze.crm_contrat c
    LEFT JOIN bronze.crm_client k ON k.client_id = c.souscripteur_id AND k._batch_id = :b
    WHERE c._batch_id = :b AND k.client_id IS NULL""")
detail["codes de type inconnus"] = lire("""SELECT c.code_type, COUNT(*) AS n FROM bronze.crm_contrat c
    LEFT JOIN bronze.crm_type_contrat t ON t.code_type = c.code_type AND t._batch_id = :b
    WHERE c._batch_id = :b AND t.code_type IS NULL GROUP BY c.code_type ORDER BY n DESC""", b=BATCH).to_dict("records")
detail["sinistres sur contrat inconnu (exemples)"] = lire("""SELECT s.sinistre_id, s.contrat_id FROM bronze.crm_sinistre s
    LEFT JOIN bronze.crm_contrat c ON c.contrat_id = s.contrat_id AND c._batch_id = :b
    WHERE s._batch_id = :b AND c.contrat_id IS NULL LIMIT 5""", b=BATCH).to_dict("records")
for k, v in detail.items():
    print(f"\n{k} : {v}")

| Relation | Lignes testées | Orphelins | Taux | Conclusion (anomalie, sentinelle, normal) |
|---|---|---|---|---|
| RQ-007 · client.pays -> ref_pays | 100 000 | 3 050 | 3.050 % | … |
| RQ-011 · contrat.code_type -> type_contrat | 100 000 | 200 | 0.200 % | … |
| RQ-012 · contrat.souscripteur_id -> client | 100 000 | 1 000 | 1.000 % | … |
| RQ-013 · contrat.assure_id -> client (sentinelle 0 exclue) | 60 000 | 300 | 0.500 % | … |
| sinistre.contrat_id -> contrat | 100 000 | 0 | 0.000 % | … |

souscripteurs orphelins distincts : 987

codes de type inconnus : [{'code_type': 'T99', 'n': 50}, {'code_type': 'A01', 'n': 50}, {'code_type': 'T13', 'n': 50}, {'code_type': 'TXX', 'n': 50}]

sinistres sur contrat inconnu (exemples) : []


In [10]:
open(f"{SODA}/checks/bronze/sentinelle_avant.yml", "w", encoding="utf-8").write("""checks for crm_contrat:
  - values in (assure_id) must exist in crm_client (client_id):
      name: RQ-013 avant traitement de la sentinelle
""")
_, avant = scan("bronze", f"{SODA}/checks/bronze/sentinelle_avant.yml")
contrats = total("SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b")
sentinelles = total("SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b AND assure_id = '0'")
orph_avant, orph_apres = int(avant[0]["valeur"]), valeur("RQ-013")
print(pd.DataFrame([{"contrats": contrats, "valeurs_sentinelle": sentinelles, "orphelins_avant": orph_avant,
                     "orphelins_apres": orph_apres, "ecart_explique_par_la_sentinelle": orph_avant - orph_apres,
                     "pct_avant": round(100 * orph_avant / contrats, 1), "pct_apres": round(100 * orph_apres / contrats, 1)}]).T.to_string(header=False))

contrats                          100000.0
valeurs_sentinelle                 40000.0
orphelins_avant                    40300.0
orphelins_apres                      300.0
ecart_explique_par_la_sentinelle   40000.0
pct_avant                             40.3
pct_apres                              0.3


In [11]:
print("| Contrôle | Orphelins | Preuve |")
print("|---|---|---|")
print(f"| Sur `assure_id`, avant traitement de la sentinelle | {fmt(orph_avant)} sur {fmt(contrats)} ({100 * orph_avant / contrats:.1f} %) | `soda/checks/bronze/sentinelle_avant.yml`, `notebooks/S3_Fatine.ipynb`, cellule 6 |")
print(f"| Sur `assure_id`, après traitement | {fmt(orph_apres)} sur {fmt(contrats - sentinelles)} contrats avec un assuré renseigné ({100 * orph_apres / contrats:.1f} % du total) | `soda/checks/bronze/contrat_checks.yml` (filtre `assure_renseigne`), cellule 6 |")
print(f"\nValeurs sentinelle '0' : {fmt(sentinelles)} ; écart expliqué par la sentinelle : {fmt(orph_avant - orph_apres)}")

| Contrôle | Orphelins | Preuve |
|---|---|---|
| Sur `assure_id`, avant traitement de la sentinelle | 40 300 sur 100 000 (40.3 %) | `soda/checks/bronze/sentinelle_avant.yml`, `notebooks/S3_Fatine.ipynb`, cellule 6 |
| Sur `assure_id`, après traitement | 300 sur 60 000 contrats avec un assuré renseigné (0.3 % du total) | `soda/checks/bronze/contrat_checks.yml` (filtre `assure_renseigne`), cellule 6 |

Valeurs sentinelle '0' : 40 000 ; écart expliqué par la sentinelle : 40 000


In [12]:
statuts = pd.DataFrame([
    ("RQ-015 · prospect titulaire d'un contrat", total("SELECT COUNT(*) FROM bronze.crm_client WHERE _batch_id = :b"), valeur("RQ-015")),
    ("RQ-016 · souscripteur sans contrat", total("SELECT COUNT(*) FROM bronze.crm_client WHERE _batch_id = :b AND type_personne = 'souscripteur'"), valeur("RQ-016"))],
    columns=["regle", "testees", "echecs"])
print(statuts.to_string(index=False))

prospects = lire("""SELECT k.client_id, k.type_personne, COUNT(*) AS nb_contrats, MIN(c.date_debut) AS premier_contrat
    FROM bronze.crm_client k JOIN bronze.crm_contrat c ON c.souscripteur_id = k.client_id AND c._batch_id = :b
    WHERE k.type_personne = 'prospect' AND k._batch_id = :b GROUP BY k.client_id, k.type_personne ORDER BY nb_contrats DESC""", b=BATCH)
print(prospects.head(10).to_string(index=False))
print("Prospects qui détiennent un contrat :", len(prospects))

sans_contrat = lire("""SELECT k.client_id, k.nom, k.prenom, k.date_creation FROM bronze.crm_client k
    LEFT JOIN bronze.crm_contrat c ON c.souscripteur_id = k.client_id AND c._batch_id = :b
    WHERE k.type_personne = 'souscripteur' AND k._batch_id = :b AND c.contrat_id IS NULL""", b=BATCH)
print(sans_contrat.head(10).to_string(index=False))

                                   regle  testees  echecs
RQ-015 · prospect titulaire d'un contrat   100000     479
      RQ-016 · souscripteur sans contrat    49898     955
client_id type_personne  nb_contrats premier_contrat
  A-28575      prospect            5      2017-10-31
  S-14559      prospect            5      2018-03-30
  A-39255      prospect            5      2016-09-29
   S-4981      prospect            5      2021-09-08
  A-19593      prospect            5      2025-09-14
  H-19157      prospect            5      2020-01-10
  H-12332      prospect            5      2019-03-21
  H-24226      prospect            5      2020-06-27
   H-2279      prospect            5      2025-08-15
  S-23095      prospect            5      2022-03-10
Prospects qui détiennent un contrat : 479
client_id      nom    prenom date_creation
  A-10025 Gaillard    Julien    2018-09-21
  A-10324   Nguyen  Nathalie    2021-04-07
  A-10382  Schmitt   Mohamed    2020-02-16
   A-1041   Dupont   Vincent 

In [13]:
rattaches = total("""SELECT COUNT(*) FROM bronze.crm_sinistre s JOIN bronze.crm_contrat c ON c.contrat_id = s.contrat_id AND c._batch_id = :b WHERE s._batch_id = :b""")
hors_periode = valeur("RQ-017")
print(f"Sinistres rattachés à un contrat : {rattaches} ; hors période de garantie (Soda) : {hors_periode} ({100 * hors_periode / rattaches:.2f} %)")

faux = lire("""SELECT COUNT(*) AS hors_periode,
       COALESCE(SUM(EXISTS (SELECT 1 FROM bronze.crm_contrat c2
                   WHERE c2._batch_id = :b AND c2.souscripteur_id = c.souscripteur_id AND c2.contrat_id <> c.contrat_id
                     AND s.date_survenance BETWEEN c2.date_debut AND c2.date_fin)), 0) AS couverts_par_un_autre_contrat
    FROM bronze.crm_sinistre s JOIN bronze.crm_contrat c ON c.contrat_id = s.contrat_id AND c._batch_id = :b
    WHERE s._batch_id = :b AND NOT (s.date_survenance BETWEEN c.date_debut AND c.date_fin)""", b=BATCH)
print(faux.to_string(index=False))

Sinistres rattachés à un contrat : 100000 ; hors période de garantie (Soda) : 1200 (1.20 %)
 hors_periode couverts_par_un_autre_contrat
         1200                           344


In [14]:
taux = lambda t, e: 100 * (t - e) / t if t else float("nan")
lignes = [
    ("RQ-015 (prospect sans contrat)", statuts.testees[0], statuts.echecs[0]),
    ("RQ-016 (souscripteur avec au moins un contrat)", statuts.testees[1], statuts.echecs[1]),
    ("RQ-017 (sinistre hors garantie)", rattaches, hors_periode),
]
print("| Règle | Couche | Lignes testées | Échecs | Taux |")
print("|---|---|---|---|---|")
for nom, t, e in lignes:
    print(f"| {nom} | BRONZE | {fmt(t)} | {fmt(e)} | {taux(t, e):.3f} % |")
print(f"\nProspects distincts qui détiennent un contrat : {len(prospects)} ; contrats concernés : {int(prospects.nb_contrats.sum())}")
print(f"Sinistres hors période couverts par un autre contrat du même souscripteur : {int(faux.couverts_par_un_autre_contrat[0])} sur {int(faux.hors_periode[0])}")

| Règle | Couche | Lignes testées | Échecs | Taux |
|---|---|---|---|---|
| RQ-015 (prospect sans contrat) | BRONZE | 100 000 | 479 | 99.521 % |
| RQ-016 (souscripteur avec au moins un contrat) | BRONZE | 49 898 | 955 | 98.086 % |
| RQ-017 (sinistre hors garantie) | BRONZE | 100 000 | 1 200 | 98.800 % |

Prospects distincts qui détiennent un contrat : 479 ; contrats concernés : 1022
Sinistres hors période couverts par un autre contrat du même souscripteur : 344 sur 1200


In [15]:
print("RQ-018 · écart entre la définition Commerciale et la fiche CD-001 (Soda) :", valeur(controle=CTRL[CTRL.controle.str.startswith("ecart_client_actif")].controle.iloc[0]), "clients")

CONDITION_CD001 = (f"_batch_id = '{BATCH}' "
                   f"AND date_debut <= '{DATE_REF}' AND date_fin >= '{DATE_REF}' "
                   f"AND date_derniere_cotisation >= '{COTISATION_MIN}' "
                   f"AND souscripteur_id IN (SELECT client_id FROM bronze.crm_client WHERE _batch_id = '{BATCH}')")
VUE = f"CREATE OR REPLACE VIEW dq.v_client_actif_bronze AS SELECT DISTINCT souscripteur_id AS client_id FROM bronze.crm_contrat WHERE {CONDITION_CD001}"
executer(VUE)
with open(f"{RACINE}/reports/S3_vue_client_actif.sql", "w", encoding="utf-8") as f:
    f.write(VUE + ";\n")

ct = lire("SELECT souscripteur_id, date_debut, date_fin, date_derniere_cotisation FROM bronze.crm_contrat WHERE _batch_id = :b", b=BATCH)
for col in ["date_debut", "date_fin", "date_derniere_cotisation"]:
    ct[col] = pd.to_datetime(ct[col], errors="coerce")
ref, cotis = pd.Timestamp(DATE_REF), pd.Timestamp(COTISATION_MIN)
en_cours = (ct.date_debut <= ref) & (ct.date_fin >= ref)
avant = {"Commerciale (au moins un contrat)": ct.souscripteur_id.nunique(),
         "Sinistres (contrat en cours)": ct[en_cours].souscripteur_id.nunique(),
         "Risques (CD-001 : en cours et cotisation à jour)": ct[en_cours & (ct.date_derniere_cotisation >= cotis)].souscripteur_id.nunique()}

unique = {d: int(lire("SELECT COUNT(*) AS n FROM dq.v_client_actif_bronze").n[0]) for d in ["Commerciale", "Sinistres", "Risques"]}
print("\nAvant, chaque direction avec sa définition :")
for d, n in avant.items():
    print(f"  {d:<50} {fmt(n)}")
print("\n| Direction | Chiffre obtenu avec la requête unique |")
print("|---|---|")
for d, n in unique.items():
    print(f"| {d} | {fmt(n)} |")
print("\nMême chiffre pour les trois directions :", len(set(unique.values())) == 1)
print("Requête enregistrée :", f"{RACINE}/reports/S3_vue_client_actif.sql")

RQ-018 · écart entre la définition Commerciale et la fiche CD-001 (Soda) : 11878 clients

Avant, chaque direction avec sa définition :
  Commerciale (au moins un contrat)                  50 508
  Sinistres (contrat en cours)                       44 116
  Risques (CD-001 : en cours et cotisation à jour)   38 049

| Direction | Chiffre obtenu avec la requête unique |
|---|---|
| Commerciale | 37 643 |
| Sinistres | 37 643 |
| Risques | 37 643 |

Même chiffre pour les trois directions : True
Requête enregistrée : /home/jovyan/work/reports/S3_vue_client_actif.sql


In [16]:
fr = lire("SELECT cp, ville FROM bronze.crm_client WHERE _batch_id = :b AND pays = 'FR'", b=BATCH)
ref = lire("SELECT code_postal, commune FROM silver.ref_postal")

brut = fr.groupby("ville").agg(codes_postaux=("cp", "nunique"), lignes=("cp", "size"))
print(brut[brut.codes_postaux > 1].sort_values("lignes", ascending=False).head(15))

multi = ref.groupby("commune").size()
print(multi[multi > 1].sort_index().head(15))

fr["code_postal"] = fr.cp.str.zfill(5)
couples = set(zip(ref.code_postal, ref.commune))
fr["couple_valide"] = [(c, v) in couples for c, v in zip(fr.code_postal, fr.ville)]
print({"adresses_france": len(fr), "couples_dans_le_referentiel": int(fr.couple_valide.sum()), "couples_absents": int((~fr.couple_valide).sum())})

print(fr[fr.ville == "Beaumont"].groupby(["cp", "ville"]).agg(lignes=("cp", "size"), couple_valide=("couple_valide", "max")).sort_values("lignes", ascending=False))

                   codes_postaux  lignes
ville                                   
Lyon                          37   13564
Villeurbanne                   3    2834
Paris                         38    2314
Bourg-en-Bresse                5    1717
Annecy                         2    1668
Vénissieux                     6    1640
Grenoble                       3    1514
Saint-Étienne                  3    1470
Caluire-et-Cuire               8    1188
Ambérieu-en-Bugey              6    1188
Chambéry                       5    1188
Oyonnax                        6    1164
Beaumont                      14    1148
Saint-Priest                   3    1137
Miribel                        8    1123
commune
Beaumont     2
Lyon         9
Marseille    3
Paris        9
dtype: int64
{'adresses_france': 95437, 'couples_dans_le_referentiel': 94755, 'couples_absents': 682}
                lignes  couple_valide
cp    ville                          
74160 Beaumont     566           True
07110 Beaumont     

In [17]:
MESURES_SODA = {
 "RQ-011": ("crm_contrat",  "SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b"),
 "RQ-012": ("crm_contrat",  "SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b"),
 "RQ-013": ("crm_contrat",  "SELECT COUNT(*) FROM bronze.crm_contrat WHERE _batch_id = :b"),
 "RQ-015": ("crm_client",   "SELECT COUNT(*) FROM bronze.crm_client WHERE _batch_id = :b"),
 "RQ-016": ("crm_client",   "SELECT COUNT(*) FROM bronze.crm_client WHERE _batch_id = :b AND type_personne = 'souscripteur'"),
 "RQ-017": ("crm_sinistre", "SELECT COUNT(*) FROM bronze.crm_sinistre s JOIN bronze.crm_contrat c ON c.contrat_id = s.contrat_id AND c._batch_id = :b WHERE s._batch_id = :b")}
for rule_id, (table, requete) in MESURES_SODA.items():
    mesurer("BRONZE", rule_id, table, total(requete), valeur(rule_id))
print(mesures("BRONZE").to_string(index=False))
print(lire("SELECT * FROM dq.dq_kpi_dimension WHERE couche = 'BRONZE'").to_string(index=False))

couche rule_id      severite      table_cible  nb_testees  nb_echecs taux_conformite seuil_min
BRONZE  RQ-001     BLOQUANTE      crm_contrat      100000        800          99.200    100.00
BRONZE  RQ-002     BLOQUANTE      crm_contrat      100000        301          99.699    100.00
BRONZE  RQ-003 AVERTISSEMENT       crm_client       49898       5989          87.998     98.00
BRONZE  RQ-005 AVERTISSEMENT       crm_client       95437       1456          98.474     98.00
BRONZE  RQ-007     BLOQUANTE       crm_client      100000       3050          96.950    100.00
BRONZE  RQ-009     BLOQUANTE       crm_client      100000        200          99.800    100.00
BRONZE  RQ-010     BLOQUANTE      crm_contrat      100000        600          99.400    100.00
BRONZE  RQ-011     BLOQUANTE      crm_contrat      100000        200          99.800    100.00
BRONZE  RQ-012     BLOQUANTE      crm_contrat      100000       1000          99.000    100.00
BRONZE  RQ-013     BLOQUANTE      crm_contrat     